# Three-Variable Mediation Experiment (X -> M -> Y)

We investigate a simple mediation model: X -> Y and X -> M -> Y.
Variables are binary: X, M, Y $\in {0, 1}$.

We will:
1. Generate synthetic data from a known SCM using `pymc` (which will depict a the real unknown SCM in the Partial Identification framework).
2. Use `ConservativePID` to bound the Total Effect (TE), Natural Direct Effect (NDE), and Natural Indirect Effect (NIE).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az

# Add parent directory to path
sys.path.append(os.path.abspath(".."))

from symbolic import Variable, Event, P, CounterfactualTerm
from inference import ConservativePID

## 1. SCM and Data Generation

**SCM Parameters:**
- $P(X=1) = p_x$
- $P(M=1 | X=x) = p_{m|x}$
- $P(Y=1 | M=m, X=x) = p_{y|m,x}$

In [ ]:
print("Running Experiment Verification...")

# True Parameters
px = 0.5
p_m_cond_x = {0: 0.2, 1: 0.7}
p_y_cond_xm = {(0, 0): 0.1, (0, 1): 0.3, (1, 0): 0.4, (1, 1): 0.8}

n_samples = 500
RANDOM_SEED = 42

print("Generating data...")
with pm.Model() as model:
    # X
    X_obs = pm.Bernoulli("X", p=px, shape=n_samples)

    # M | X
    p_m = pm.math.switch(X_obs, p_m_cond_x[1], p_m_cond_x[0])
    M_obs = pm.Bernoulli("M", p=p_m, shape=n_samples)

    # Y | X, M
    # probs_y = np.array(
    #     [
    #         p_y_cond_xm[(0, 0)],
    #         p_y_cond_xm[(0, 1)],
    #         p_y_cond_xm[(1, 0)],
    #         p_y_cond_xm[(1, 1)],
    #     ]
    # )
    # p_y = probs_y[idx]

    # Use nested switch for safe symbolic indexing
    p_y_x1 = pm.math.switch(M_obs, p_y_cond_xm[(1, 1)], p_y_cond_xm[(1, 0)])
    p_y_x0 = pm.math.switch(M_obs, p_y_cond_xm[(0, 1)], p_y_cond_xm[(0, 0)])
    p_y = pm.math.switch(X_obs, p_y_x1, p_y_x0)

    Y_obs = pm.Bernoulli("Y", p=p_y, shape=n_samples)

    trace = pm.sample_prior_predictive(samples=1, random_seed=RANDOM_SEED)

# Extract data
df = pd.DataFrame(
    {
        "X": trace.prior["X"].values.flatten(),
        "M": trace.prior["M"].values.flatten(),
        "Y": trace.prior["Y"].values.flatten(),
    }
)

df.head()

Sampling: [M, X, Y]


Running Experiment Verification...
Generating data...


,X,M,Y
0,1,1,1
1,1,1,1
2,1,1,1
3,0,0,0
4,1,0,1


## 2. Prepare Observational Data
Calculate empirical $P(X, M, Y)$.

In [8]:
observational_data = {}
total = len(df)

for x in [0, 1]:
    for m in [0, 1]:
        for y in [0, 1]:
            count = len(df[(df["X"] == x) & (df["M"] == m) & (df["Y"] == y)])
            observational_data[(x, m, y)] = count / total

print("Observational Data:", observational_data)

Observational Data: {(0, 0, 0): 0.364, (0, 0, 1): 0.026, (0, 1, 0): 0.072, (0, 1, 1): 0.026, (1, 0, 0): 0.12, (1, 0, 1): 0.07, (1, 1, 0): 0.062, (1, 1, 1): 0.26}


## 3. Initialize ConservativePID

In [9]:
X_var = Variable("X", (0, 1))
M_var = Variable("M", (0, 1))
Y_var = Variable("Y", (0, 1))
variables = [X_var, M_var, Y_var]

# Note: ConservativePID expects data keys to match the variable order passed in 'variables'.
# Currently keys are (x, m, y) which matches [X_var, M_var, Y_var].

cpid = ConservativePID(variables, observational_data)

## 4. Define Truth Calculations (for Validation)
We calculate the true causal effects from the known SCM parameters.

In [10]:
def get_true_prob(x_val, m_intervention_x=None):
    # Calculate P(Y=1) under interventions
    # If m_intervention_x is None, M follows X=x_val naturally: M ~ P(M|X=x_val)
    # If m_intervention_x is set (0 or 1), M follows X=m_intervention_x: M ~ P(M|X=m_intervention_x)

    x_for_m = x_val if m_intervention_x is None else m_intervention_x

    prob_y1 = 0.0
    # Sum over M
    for m in [0, 1]:
        p_m = p_m_cond_x[x_for_m] if m == 1 else (1 - p_m_cond_x[x_for_m])
        p_y = p_y_cond_xm[(x_val, m)]  # P(Y=1 | X=x_val, M=m)
        prob_y1 += p_y * p_m
    return prob_y1


# TE: P(Y_{X=1}=1) - P(Y_{X=0}=1)
true_te = get_true_prob(1, None) - get_true_prob(0, None)

# NDE: P(Y_{X=1, M_{X=0}}=1) - P(Y_{X=0, M_{X=0}}=1)  (which is just P(Y_{X=0}=1))
# Wait, P(Y_{X=0, M_{X=0}}=1) == P(Y_{X=0}=1) is true.
# Calculate term 1: X for Y is 1, X for M is 0
true_nde = get_true_prob(1, 0) - get_true_prob(0, 0)

# NIE: P(Y_{X=0, M_{X=1}}=1) - P(Y_{X=0, M_{X=0}}=1)
true_nie = get_true_prob(0, 1) - get_true_prob(0, 0)

print(f"True TE: {true_te:.4f}")
print(f"True NDE: {true_nde:.4f}")
print(f"True NIE: {true_nie:.4f}")

True TE: 0.5400
True NDE: 0.3400
True NIE: 0.1000


## 5. Compute Bounds

In [11]:
# Total Effect
# Query: P(Y_{X=1} = 1) - P(Y_{X=0} = 1)
q_te = P(Y_var @ {X_var: 1} == 1) - P(Y_var @ {X_var: 0} == 1)
lb_te, ub_te = cpid.infer(q_te)
print(f"TE Bounds: [{lb_te:.4f}, {ub_te:.4f}]")

# NDE
# Query: P(Y_{X=1, M_{X=0}} = 1) - P(Y_{X=0} = 1)

# Construct nested term: M_{X=0}
# In the syntax: M_var @ {X_var: 0}
# Construct Y term: Y_{X=1, M=M_{X=0}}
nested_M = M_var @ {X_var: 0}
y_nde_term = Y_var @ {X_var: 1, M_var: nested_M}
# Reference term is just Y_{X=0}
y_ref_term = Y_var @ {X_var: 0}

# Or alternatively Y_{X=0, M_{X=0}}, which should match Y_{X=0}

q_nde = P(y_nde_term == 1) - P(y_ref_term == 1)
lb_nde, ub_nde = cpid.infer(q_nde)
print(f"NDE Bounds: [{lb_nde:.4f}, {ub_nde:.4f}]")

# NIE
# Query: P(Y_{X=0, M_{X=1}} = 1) - P(Y_{X=0} = 1)
nested_M_nie = M_var @ {X_var: 1}
y_nie_term = Y_var @ {X_var: 0, M_var: nested_M_nie}

q_nie = P(y_nie_term == 1) - P(y_ref_term == 1)
lb_nie, ub_nie = cpid.infer(q_nie)
print(f"NIE Bounds: [{lb_nie:.4f}, {ub_nie:.4f}]")

2026-01-27 20:26:01.735 | INFO     | inference:infer:83 - Found 3 valid causal orders compatible with the query.
2026-01-27 20:26:01.735 | INFO     | inference:infer:95 - Processing Order 1/3: ['M', 'X', 'Y']
2026-01-27 20:26:01.736 | INFO     | canonical:__init__:42 - Generated Basis with 128 worlds.
2026-01-27 20:26:01.804 | INFO     | inference:infer:95 - Processing Order 2/3: ['X', 'Y', 'M']
2026-01-27 20:26:01.805 | INFO     | canonical:__init__:42 - Generated Basis with 128 worlds.
2026-01-27 20:26:01.844 | INFO     | inference:infer:95 - Processing Order 3/3: ['X', 'M', 'Y']
2026-01-27 20:26:01.844 | INFO     | canonical:__init__:42 - Generated Basis with 128 worlds.
2026-01-27 20:26:01.884 | INFO     | inference:infer:83 - Found 1 valid causal orders compatible with the query.
2026-01-27 20:26:01.885 | INFO     | inference:infer:95 - Processing Order 1/1: ['X', 'M', 'Y']
2026-01-27 20:26:01.885 | INFO     | canonical:__init__:42 - Generated Basis with 128 worlds.
2026-01-27 20:

TE Bounds: [-0.2880, 0.7700]
NDE Bounds: [-0.5640, 0.9480]
NIE Bounds: [-0.5640, 0.9480]


## 6. Verification

In [12]:
def check_bounds(name, lb, ub, true_val):
    contains = lb <= true_val <= ub
    status = "PASS" if contains else "FAIL"
    print(f"{name}: True={true_val:.4f} in [{lb:.4f}, {ub:.4f}] ? {status}")


check_bounds("TE", lb_te, ub_te, true_te)
check_bounds("NDE", lb_nde, ub_nde, true_nde)
check_bounds("NIE", lb_nie, ub_nie, true_nie)

TE: True=0.5400 in [-0.2880, 0.7700] ? PASS
NDE: True=0.3400 in [-0.5640, 0.9480] ? PASS
NIE: True=0.1000 in [-0.5640, 0.9480] ? PASS
